## 05 — Prophet
Huấn luyện Facebook Prophet, dự báo Q2/2026 và đánh giá qua Cross-Validation.

In [ ]:
# Huấn luyện Prophet
m = Prophet(
    holidays                = holidays,
    seasonality_mode        = seasonality_mode,
    seasonality_prior_scale = seasonality_prior_scale,
    holidays_prior_scale    = holidays_prior_scale,
    changepoint_prior_scale = changepoint_prior_scale,
    yearly_seasonality      = True,
    weekly_seasonality      = False,
    daily_seasonality       = False,
)
m.fit(training)
print("Prophet fit xong.")

In [ ]:
# Dự báo Q2/2026
future_prophet    = m.make_future_dataframe(periods=3, freq="MS")
forecast_prophet  = m.predict(future_prophet)
pred_prophet      = forecast_prophet[["ds","yhat","yhat_lower","yhat_upper"]].tail(3).reset_index(drop=True)
pred_prophet["tháng"] = pred_prophet["ds"].dt.strftime("T%m/%Y")

print("Dự báo Prophet Q2/2026:")
for _, r in pred_prophet.iterrows():
    print(f"  {r['tháng']}: {r['yhat']/1e9:.2f} tỷ VND  [{r['yhat_lower']/1e9:.2f} – {r['yhat_upper']/1e9:.2f}]")

In [ ]:
# Biểu đồ thành phần
fig = m.plot_components(forecast_prophet)
fig.tight_layout()
plt.show()

In [ ]:
# Cross-Validation
df_cv   = cross_validation(m, horizon="90 days", period="30 days", initial="90 days")
metrics = performance_metrics(df_cv)
print(f"RMSE  : {metrics['rmse'].mean()/1e9:.3f} tỷ")
print(f"MAPE  : {metrics['mape'].mean()*100:.1f}%")